In [1]:
import pandas as pd
import pickle
import numpy as np

with open('../output/qrf-final.pkl', 'rb+') as f:
    qrf_model = pickle.load(f)

In [2]:
print('max depth', qrf_model.max_depth)
print('max samples', qrf_model.max_samples)
print('max features', qrf_model.max_features)
print('max samples leaf', qrf_model.max_samples_leaf)

max depth 16
max samples 0.75
max features 0.75
max samples leaf 1


In [3]:
in_pi_mask = np.load('../output/in_pi_mask_val.npy', allow_pickle=True)
val_mmsis = np.load('../val-mmsis-test.npy', allow_pickle=True)
val_days = np.load('../val-days-test.npy', allow_pickle=True)

group_results = {}
for ix, (mmsi, day) in enumerate(zip(val_mmsis, val_days)):
    
    if (mmsi, day) not in group_results:
        group_results[(mmsi, day)] = [ in_pi_mask[ix] ]
    else:
        group_results[(mmsi, day)].append( in_pi_mask[ix] )

df = []


for (mmsi, day), pi_masks in group_results.items():
    means = np.array(pi_masks)\
        .reshape((-1, 25, 2))\
        .mean(axis=1)\
        .mean(axis=0)

    df.append({
        'mmsi': mmsi,
        'day': day,
        'delta_cog': means[0],
        'delta_dif': means[1],
    })

df = pd.DataFrame(df)
thresh_delta_cog = np.quantile(df['delta_cog'], 0.05)
thresh_delta_dif = np.quantile(df['delta_dif'], 0.05)

print( 'delta_cog limit:', np.quantile(df['delta_cog'], 0.05) )
print( 'delta_dif liimt:', np.quantile(df['delta_dif'], 0.05) )

delta_cog limit: 0.8370961420698105
delta_dif liimt: 0.8595311125078572


In [4]:
in_pi_mask = np.load('../output/in_pi_mask_test.npy', allow_pickle=True)
test_mmsis = np.load('../all-mmsis-test.npy', allow_pickle=True)
test_days = np.load('../all-days-test.npy', allow_pickle=True)

group_results = {}
for ix, (mmsi, day) in enumerate(zip(test_mmsis, test_days)):
    
    if (mmsi, day) not in group_results:
        group_results[(mmsi, day)] = [ in_pi_mask[ix] ]
    else:
        group_results[(mmsi, day)].append( in_pi_mask[ix] )

df = []


for (mmsi, day), pi_masks in group_results.items():
    means = np.array(pi_masks)\
        .reshape((-1, 25, 2))\
        .mean(axis=1)\
        .mean(axis=0)

    df.append({
        'mmsi': mmsi,
        'day': day,
        'delta_cog': means[0],
        'delta_dif': means[1],
    })

df = pd.DataFrame(df)
print( np.mean( df['delta_cog'] < thresh_delta_cog ) )
print( np.mean( df['delta_dif'] < thresh_delta_dif ) )

0.06769230769230769
0.055384615384615386


In [6]:
X_arr = np.load('../anom-x.npy', allow_pickle=True)
reshape_params_x = (-1, 25*X_arr.shape[2])

y_pred = qrf_model.predict(
    X_arr.reshape(reshape_params_x),
    quantiles=[0.025, 0.975]
)
y_low = y_pred[:, :, 0]
y_high = y_pred[:, :, 1]


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.0s finished


In [10]:
y_low = y_low.reshape((-1, 25, 2))
y_high = y_high.reshape((-1, 25, 2))

y_arr = np.load('../anom-y.npy', allow_pickle=True)
np.mean(
    (y_low[:, 0, 0] < y_arr[:, 0, 0]) &
    (y_high[:, 0, 0] > y_arr[:, 0, 0])
)

np.float64(0.8822254335260116)

In [11]:
np.mean(
    (y_low[:, 0, 1] < y_arr[:, 0, 1]) &
    (y_high[:, 0, 1] > y_arr[:, 0, 1])
)

np.float64(0.833092485549133)

In [12]:
X_test = np.load('../X_test_final.npy')
y_test = np.load('../y_test_final.npy')

In [14]:
y_pred = qrf_model.predict(X_test.reshape((-1, 25*X_test.shape[2])), quantiles=[0.025, 0.975])

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.2s finished


In [17]:
y_low = y_pred[:, :, 0]
y_low = y_low.reshape((-1, 25, 2))
y_high = y_pred[:, :, 1]
y_high = y_high.reshape((-1, 25, 2))

In [20]:
np.mean( (y_test < y_high) & (y_low < y_test), axis=0).mean(axis=0)

array([0.91275513, 0.93577854])

In [18]:
np.mean( np.abs(y_high - y_low), axis=0).mean(axis=0)

array([0.26982665, 0.13797538])